# PFR 03 — Lead screen: which catalog features predict the perp return AFTER t?

Protocol: `discussions/Pairwise_Feature_Regression/007_claude_lead_screen_preregistration.md`.

For each PFR 01 catalog feature $X_t$ and $h\in\{5,15,60\}$ min: partial IC of $X_t$ with the Deribit perp forward return $y_{t,h}$ after removing the current perp return. Discovery = first 60% of UTC days, confirmation = last 40%. Whole-day bootstrap. A cell is **confirmed** if discovery passes Bonferroni (features × horizons, per asset) and confirmation has the same sign at one-sided 5%. Decile long–short (bp) in confirmation is compared with a 12 bp round trip.

In [ ]:
import os,sys,json,importlib,unittest,importlib.util
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=next((p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src'/'data_engineering').is_dir()),None)
assert ROOT is not None,'Run inside the repository'
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import pairwise_feature_lead as pl
from data_engineering.modeling import pairwise_feature_regression as pfr
for m in (pfr,pl):importlib.reload(m)
assert pl.VERSION=='PFR03-0.1.0',pl.VERSION
cfg=pl.Specification()
SYNTHETIC=os.getenv('PFR03_SYNTHETIC','0')=='1'
OUT=ROOT/'notebooks'/'Pairwise_Feature_Regression'/'03'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='pfr03-');OUT=Path(tmp.name)/'03'
OUT.mkdir(parents=True,exist_ok=True)
# in-kernel tests (Windows/Conda-safe, as PFR 01)
spec_t=importlib.util.spec_from_file_location('_pfr03_tests',ROOT/'tests'/'test_pairwise_feature_lead.py')
tm=importlib.util.module_from_spec(spec_t);spec_t.loader.exec_module(tm)
import pytest
rc=pytest.main(['-q',str(ROOT/'tests'/'test_pairwise_feature_lead.py')])
assert rc==0,'PFR03 tests failed'
print('PFR03',pl.VERSION,'synthetic:',SYNTHETIC,'output:',OUT)

In [ ]:
ASSET_DATA={}
if SYNTHETIC:
    for i,(a,b) in enumerate((('BTC',.12),('ETH',.0))):
        f,t,r=pl.synthetic_states(n_days=28,seed=10+i,beta_lead=b,bounce=-.1)
        ASSET_DATA[a]=(f,t,r)
    source='SYNTHETIC'
else:
    import duckdb
    from data_engineering.catalog.database import RESEARCH_DATABASE
    from data_engineering.modeling.full_information import FEATURE_SPECS
    q=lambda n:'"'+n.replace('"','""')+'"'
    cols=list(dict.fromkeys(['underlying','state_time','expiry_date','utc_minute_of_day','session_progress_fraction',
        'settlement_locked_fraction','is_weekend','minutes_to_expiry','in_settlement_window','perp_integrity_ok',
        'microstructure_features_ready','instability_features_ready','option_atm_quote_ok','option_shape_quote_ok',
        'fit_ok','option_snapshot_ready']+[s.column for s in FEATURE_SPECS]+[e[1] for e in pfr.EXTRAS]))
    with duckdb.connect(str(RESEARCH_DATABASE),read_only=True) as con:
        con.execute("SET TimeZone='UTC'")
        fields=set(con.execute('DESCRIBE SELECT * FROM time_series_feature_state').df().column_name)
        want=[c for c in cols if c in fields]
        sql='SELECT '+', '.join(map(q,want))+' FROM time_series_feature_state WHERE underlying IN ('+','.join('?'*len(cfg.assets))+') ORDER BY underlying, state_time'
        states=con.execute(sql,list(cfg.assets)).df()
    states['state_time']=pd.to_datetime(states.state_time,utc=True)
    assert not states.duplicated(['underlying','state_time']).any()
    last_full=states.state_time.max().floor('D')
    states=states[states.state_time<last_full]                      # complete UTC days only
    print('States:',len(states),states.state_time.min(),'->',states.state_time.max())
    for a,part in states.groupby('underlying'):
        part=part.reset_index(drop=True)
        feats,inv=pfr.model_feature_frame(part)
        ASSET_DATA[a]=(feats,part.state_time,feats['perp_ret_5m'])   # integrity-gated perp return is the target
    source=f'research.duckdb time_series_feature_state, {cfg.assets}'
print({a:len(v[0]) for a,v in ASSET_DATA.items()})

In [ ]:
tables=[];metas={}
for a,(f,t,r) in ASSET_DATA.items():
    tab,meta=pl.screen_asset(f,t,r,cfg)
    tables.append(tab.assign(asset=a));metas[a]=meta;print(a,meta)
table=pd.concat(tables,ignore_index=True)
cols=['asset','feature','horizon_min','disc_raw_ic','disc_partial_ic','conf_raw_ic','conf_partial_ic','conf_partial_lo','conf_partial_hi',
      'conf_partial_n','discovery_pass','confirmed','conf_long_short_bp','covers_cost']
print('Confirmed cells:')
display(table[table.confirmed][cols].sort_values(['horizon_min','asset']).round(4))
print('Discovery passes that failed confirmation:')
display(table[table.discovery_pass&~table.confirmed][cols].round(4))

In [ ]:
print('Largest |confirmation partial IC| per asset/horizon (descriptive, not a selection rule):')
t2=table[table.enough_data].copy();t2['abs']=t2.conf_partial_ic.abs()
display(t2.sort_values('abs',ascending=False).groupby(['asset','horizon_min']).head(5)[cols].sort_values(['asset','horizon_min']).round(4))
summ=pl.summary(table,cfg);display(summ)
dec=pl.decision(table,cfg,synthetic=SYNTHETIC);print(json.dumps(dec,indent=2,default=str))

In [ ]:
table.to_csv(OUT/'03_lead_screen.csv',index=False);summ.to_csv(OUT/'03_cross_asset_summary.csv',index=False)
(OUT/'03_decision.json').write_text(json.dumps(dec,indent=2,default=str),encoding='utf-8')
(OUT/'03_manifest.json').write_text(json.dumps({'module':'PFR03','registration':pl.registration(),'source':source,
    'synthetic':SYNTHETIC,'split':metas},indent=2,default=str),encoding='utf-8')
print('Saved 4 outputs to',OUT)